In [2]:
!pip install pandas numpy matplotlib seaborn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import os
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────
BASE_DIR      = Path().resolve().parent
RAW_DIR       = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR :", BASE_DIR)
print("RAW_DIR  :", RAW_DIR)
print("Files    :", os.listdir(RAW_DIR))

BASE_DIR : C:\Users\theyo\OneDrive\Desktop\DS task\MYNA-Myntra-AI-Stylist
RAW_DIR  : C:\Users\theyo\OneDrive\Desktop\DS task\MYNA-Myntra-AI-Stylist\data\raw
Files    : ['images', 'myntra202305041052.csv', 'myntra202305041052.csv.zip', 'styles.csv', 'styles.csv.zip', 'user_uploads']


In [4]:
# Check styles.csv for formal shirts
styles_zip = RAW_DIR / "styles.csv.zip"
styles_csv = RAW_DIR / "styles.csv"

if styles_zip.exists() and not styles_csv.exists():
    with zipfile.ZipFile(styles_zip, 'r') as z:
        z.extractall(RAW_DIR)

styles = pd.read_csv(styles_csv, on_bad_lines='skip')
print(f"Shape: {styles.shape}")
print(f"Columns: {list(styles.columns)}")
print(f"\narticleType values:")
print(styles['articleType'].value_counts().head(30))
print(f"\nFormal shirts:")
formal = styles[styles['articleType'].str.lower().str.contains(
    'shirt', na=False)]
print(formal[['gender','articleType','baseColour',
              'usage','productDisplayName']].head(20))

Shape: (44441, 10)
Columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName']

articleType values:
articleType
Tshirts                  7069
Shirts                   3215
Casual Shoes             2846
Watches                  2542
Sports Shoes             2036
Kurtas                   1844
Tops                     1762
Handbags                 1759
Heels                    1323
Sunglasses               1073
Wallets                   936
Flip Flops                916
Sandals                   897
Briefs                    849
Belts                     813
Backpacks                 724
Socks                     686
Formal Shoes              637
Perfume and Body Mist     614
Jeans                     608
Shorts                    547
Trousers                  530
Flats                     500
Bra                       477
Dresses                   464
Sarees                    427
Earrings                  417
D

In [5]:
print("All article types in styles.csv:")
print(styles['articleType'].value_counts().to_string())

print("\nAll usage types:")
print(styles['usage'].value_counts())

print("\nAll gender types:")
print(styles['gender'].value_counts())

All article types in styles.csv:
articleType
Tshirts                      7069
Shirts                       3215
Casual Shoes                 2846
Watches                      2542
Sports Shoes                 2036
Kurtas                       1844
Tops                         1762
Handbags                     1759
Heels                        1323
Sunglasses                   1073
Wallets                       936
Flip Flops                    916
Sandals                       897
Briefs                        849
Belts                         813
Backpacks                     724
Socks                         686
Formal Shoes                  637
Perfume and Body Mist         614
Jeans                         608
Shorts                        547
Trousers                      530
Flats                         500
Bra                           477
Dresses                       464
Sarees                        427
Earrings                      417
Deodorant                     347
Nai

In [6]:
styles_clean = styles.copy()

# Rename columns
styles_clean = styles_clean.rename(columns={
    'id'                 : 'style_id',
    'productDisplayName' : 'name',
    'baseColour'         : 'color',
    'articleType'        : 'article_type',
    'subCategory'        : 'sub_category',
    'masterCategory'     : 'master_category',
})

# Lowercase name
styles_clean['name'] = styles_clean['name'].fillna('').str.lower()

# Fix gender — separate Boys/Girls from Men/Women
def fix_gender(g):
    if pd.isna(g): return 'Unisex'
    g = str(g).strip()
    if g == 'Boys': return 'Boys'
    if g == 'Girls': return 'Girls'
    if g == 'Women': return 'Women'
    if g == 'Men': return 'Men'
    return 'Unisex'

styles_clean['gender'] = styles_clean['gender'].apply(fix_gender)

# Map articleType → component_type
def map_article_to_component(article):
    if pd.isna(article): return 'Other'
    a = str(article).lower()
    if any(w in a for w in ['shirt', 'kurta', 'kurti', 'top',
                             'tshirt', 't-shirt', 'blouse', 'saree',
                             'lehenga', 'hoodie', 'jacket', 'sweater',
                             'blazer', 'coat', 'dress', 'gown',
                             'sherwani', 'jumpsuit', 'romper',
                             'sweatshirt', 'cardigan', 'vest', 'tunic',
                             'indo western', 'nehru jackets',
                             'dupatta', 'salwar']):
        return 'Top'
    elif any(w in a for w in ['jeans', 'trouser', 'pant', 'dhoti',
                               'skirt', 'shorts', 'legging', 'trackpant',
                               'capri', 'tights', 'churidar', 'brief',
                               'trunk', 'boxer', 'swimwear']):
        return 'Bottom'
    elif any(w in a for w in ['shoe', 'sandal', 'heel', 'boot',
                               'sneaker', 'slipper', 'loafer', 'flat',
                               'wedge', 'mule', 'flip flop', 'clog',
                               'formal shoes', 'sports shoes',
                               'casual shoes']):
        return 'Footwear'
    elif any(w in a for w in ['bag', 'watch', 'belt', 'jewel',
                               'necklace', 'earring', 'ring', 'bangle',
                               'sunglass', 'wallet', 'cap', 'hat',
                               'scarf', 'stole', 'sock', 'tie',
                               'backpack', 'clutch', 'handbag',
                               'perfume', 'bracelet', 'anklet']):
        return 'Accessories'
    return 'Other'

styles_clean['component_type'] = styles_clean['article_type'].apply(
    map_article_to_component)

# Drop non-fashion rows
styles_clean = styles_clean[
    styles_clean['component_type'] != 'Other']

# Keep only adults
styles_clean = styles_clean[
    styles_clean['gender'].isin(['Men', 'Women', 'Unisex'])]

# Add usage column (casual/formal/sports/ethnic)
styles_clean['usage'] = styles_clean['usage'].fillna('Casual')

print(f"Clean styles shape: {styles_clean.shape}")
print(f"\nComponent types:")
print(styles_clean['component_type'].value_counts())
print(f"\nGender split:")
print(styles_clean['gender'].value_counts())
print(f"\nUsage types:")
print(styles_clean['usage'].value_counts())

Clean styles shape: (39480, 11)

Component types:
component_type
Top            15838
Accessories    11084
Footwear        9132
Bottom          3426
Name: count, dtype: int64

Gender split:
gender
Men       21426
Women     16004
Unisex     2050
Name: count, dtype: int64

Usage types:
usage
Casual          30134
Sports           3922
Ethnic           3124
Formal           2185
Smart Casual       67
Party              28
Travel             20
Name: count, dtype: int64


In [7]:
# Shirts by usage in styles.csv
shirts_by_usage = styles_clean[
    styles_clean['article_type'].str.lower() == 'shirts'
]['usage'].value_counts()
print("Shirts by usage in styles.csv:")
print(shirts_by_usage)

# Formal shirts specifically
formal_shirts = styles_clean[
    (styles_clean['article_type'].str.lower() == 'shirts') &
    (styles_clean['usage'].str.lower() == 'formal')
]
print(f"\nFormal shirts count: {len(formal_shirts)}")
print(formal_shirts[['gender','name','color']].head(15))

Shirts by usage in styles.csv:
usage
Casual          2237
Formal           866
Smart Casual      18
Ethnic            13
Party              2
Name: count, dtype: int64

Formal shirts count: 866
     gender                                      name      color
15      Men     reid & taylor men check purple shirts     Purple
30      Men          john players men navy blue shirt  Navy Blue
48      Men  john miller men stripes white red shirts        Red
382     Men      indigo nation men grey striped shirt       Grey
433     Men         john miller men red striped shirt        Red
503     Men    peter england men stripes maroon shirt     Maroon
698   Women                    arrow woman blue shirt       Blue
782     Men      reid & taylor men solid cream shirts      Cream
784     Men         mark taylor men plain blue shirts       Blue
822     Men               john miller men white shirt      White
921     Men      peter england men stripes blue shirt       Blue
1000    Men             re

In [11]:
# Keep only useful columns
cols_to_keep = [
    'style_id', 'name', 'gender', 'master_category',
    'sub_category', 'article_type', 'component_type',
    'color', 'season', 'usage'
]
styles_final = styles_clean[cols_to_keep].copy()
styles_final = styles_final.reset_index(drop=True)

out_path = PROCESSED_DIR / "clean_styles.csv"
styles_final.to_csv(out_path, index=False)

print("=" * 50)
print("STYLES.CSV CLEANED AND SAVED")
print("=" * 50)
print(f"Total products : {len(styles_final):,}")
print(f"Saved to       : {out_path}")
print(f"Columns        : {list(styles_final.columns)}")

STYLES.CSV CLEANED AND SAVED
Total products : 39,480
Saved to       : C:\Users\theyo\OneDrive\Desktop\DS task\MYNA-Myntra-AI-Stylist\data\processed\clean_styles.csv
Columns        : ['style_id', 'name', 'gender', 'master_category', 'sub_category', 'article_type', 'component_type', 'color', 'season', 'usage']
